# Movies IIS

Formally, we define our system as a triple: **$\mathcal{I} = \langle \mathcal{G}, \mathcal{S}, \mathcal{M} \rangle$**. 
* **$\mathcal{S}$** represents our heterogeneous source schemas.
* **$\mathcal{G}$** is the unified global schema presented to the user.
* **$\mathcal{M}$** is the set of mapping assertions that bridge the two.

## **Source Schema**

The Source Schema ($\mathbf{S}$) describes the structure of the data sources available for the project.

| Source Table | Attributes | Primary Key (PK) | Notes |
| :--- | :--- | :--- | :--- |
| **$\text{Box\_Office}$** | $\text{title}$, $\text{year}$, $\text{revenue\_worldwide}$, $\text{revenue\_domestic}$ | $(\text{title}, \text{year})$ | CSV Source containing financial data. |
| **$\text{TMDB\_Movies}$** | $\text{movie\_id}$, $\text{title}$, $\text{year}$, $\text{budget}$, $\text{revenue\_worldwide}$, $\text{vote\_count}$, $\text{vote\_average}$ | $\text{movie\_id}$ | JSON/CSV Source. Contains metadata. |
| **$\text{TMDB\_Cast}$** | $\text{movie\_id}$, $\text{actor\_name}$ | $(\text{movie\_id}, \text{actor\_name})$ | Bridge table extracted from TMDB JSON. |
| **$\text{TMDB\_Genres}$** | $\text{movie\_id}$, $\text{genre\_name}$ | $(\text{movie\_id}, \text{genre\_name})$ | Bridge table extracted from TMDB JSON. |
| **$\text{IMDB\_Cast}$** | $\text{actor\_name}$, $\text{birth\_year}$, $\text{death\_year}$ | $\text{actor\_name}$ | CSV Source. Contains biographical data. |

## **Global Schema**

The Global Schema ($\mathbf{G}$) represents the reconciled view of the data. We utilize **Natural Keys** to facilitate the integration of data from sources that lack shared synthetic identifiers.

| Global Table | Attributes | Primary Key (PK) | Foreign Keys (FK) | Derivation/Notes |
| :--- | :--- | :--- | :--- | :--- |
| **$\text{Movies}$** | $\text{title}$, $\text{year}$, $\text{tmdb\_id}$, $\text{vote\_count}$, $\text{vote\_average}$, $\text{budget}$, $\text{revenue\_worldwide}$, $\text{revenue\_domestic}$, $\text{profit}$ | $(\text{title}, \text{year})$ | | $\text{tmdb\_id}$ is kept as an attribute (nullable). $\text{revenue\_worldwide}$ merges data from both sources. |
| **$\text{Actors}$** | $\text{actor\_name}$, $\text{birth\_year}$, $\text{death\_year}$ | $\text{actor\_name}$ | | We adopt the **Unique Name Assumption** for actors to allow linking across sources without common IDs. |
| **$\text{Genres}$** | $\text{genre\_name}$ | $\text{genre\_name}$ | | Simple lookup table for genres. |
| **$\text{Movie\_Actors}$** | $\text{title}$, $\text{year}$, $\text{actor\_name}$ | $(\text{title}, \text{year}, \text{actor\_name})$ | $(\text{title}, \text{year}) \to \text{Movies}$<br>$\text{actor\_name} \to \text{Actors}$ | Bridge table. Links are established via the natural keys. |
| **$\text{Movie\_Genres}$** | $\text{title}$, $\text{year}$, $\text{genre\_name}$ | $(\text{title}, \text{year}, \text{genre\_name})$ | $(\text{title}, \text{year}) \to \text{Movies}$<br>$\text{genre\_name} \to \text{Genres}$ | Bridge table. |

---

## **Mapping Assertions (M)**

This section formalizes the relationship between $\mathbf{S}$ and $\mathbf{G}$ using GAV (Global-As-View). We utilize a **Union approach** for the `Movies` relation, creating a global entry if a movie exists in *either* TMDB or BoxOffice.

#### A. Mappings for Movies (Union Approach)

*   **M1: TMDB to Movie (Core Metadata)** 
    
    Maps core movie attributes from TMDB to the Global Schema.    
    > $$\forall i, t, y, b, r_w, v_c, v_a (\text{TMDB\_Movies}(i, t, y, b, r_w, v_c, v_a) \rightarrow \exists r_d, p (\text{Movies}(t, y, i, v_c, v_a, b, r_w, r_d, p)))$$

*   **M2: BoxOffice to Movie (Financial Data)**
    
    Maps financial data from BoxOffice.

    > $$\forall t, y, r_w, r_d (\text{Box\_Office}(t, y, r_w, r_d) \rightarrow \exists i, v_c, v_a, b, p (\text{Movies}(t, y, i, v_c, v_a, b, r_w, r_d, p)))$$

    ***Note**: To handle the overlap between $M_1$ and $M_2$, we adopt a **Source Preference Resolution Policy**. In the event of a conflict on the revenue_worldwide attribute for the same natural key $(t, y)$, the value from $S_{BoxOffice}$ is prioritized to maintain Global Consistency.*

#### B. Mapping for Actors (GAV)

*   **M3: IMDB to Actors**
    
    Populates biographical data.

    >$$
    >\forall n, b, d (\text{IMDB\_Cast}(n, b, d) \rightarrow \text{Actors}(n, b, d))
    >$$

#### C. Mapping for Genres (GAV)

* **M4: Populating Genres Dictionary**
    
    Extracts the list of unique genres from the source.

    >$$
    >\forall i, g (\text{TMDB\_Genres}(i, g) \rightarrow \text{Genres}(g))
    >$$

#### D. Mappings for Bridge Tables (Join GAV)

*   **M5: Populating Movie_Actors**
    
    This mapping populates the Movie_Actors relation by performing a *cross-source reconciliation*. 
    * I join $\text{TMDB\_Movies}$ (to retrieve the natural key: Title/Year) with $\text{TMDB\_Cast}$ (to identify the movie-actor relationship). 
    * Crucially, I then join this result with the Global Actors relation (previously populated via IMDB) to add birth and death years info. 
    * Ensure bridge between movies and actors that are effectively present in the global instance.

    >$$
    >\forall t, y, n, i (\text{TMDB\_Movies}(i, t, y, \dots) \land \text{TMDB\_Cast}(i, n) \land \text{Actors}(n, \cdots)\rightarrow \text{Movie\_Actors}(t, y, n))
    >$$

*   **M6: Populating Movie_Genres**
    
    Links movies and genres.

    >$$
    >\forall t, y, g, i (\text{TMDB\_Movies}(i, t, y, \dots) \land \text{TMDB\_Genres}(i, g) \rightarrow \text{Movie\_Genres}(t, y, g))
    >$$

## **Global Schema Constraints**

These constraints ($\mathbf{C}_{\mathbf{G}}$) ensure data consistency within the Global Schema.

### I. Relational Model Constraints

#### C1: Primary Key Constraints
We utilize **Natural Keys** to ensure uniqueness across the integrated view.

1.  **Movies:** $\text{PK}(\text{Movies}) = (\text{title}, \text{year})$
2.  **Actors:** $\text{PK}(\text{Actors}) = (\text{actor\_name})$
3.  **Genres:** $\text{PK}(\text{Genres}) = (\text{genre\_name})$
4.  **Movie_Actors:** $\text{PK}(\text{Movie\_Actors}) = (\text{title}, \text{year}, \text{actor\_name})$
5.  **Movie_Genres:** $\text{PK}(\text{Movie\_Genres}) = (\text{title}, \text{year}, \text{genre\_name})$

#### C1.1: FOL Primary Key Constraints

Primary Key constraints (uniqueness) are formally defined in FOL by stating that if two atoms share the same key values, all other attributes in those atoms must be identical. Due to the high arity of the global relations, we provide the representative formalization for the core entities:

1. Movies (Key: title, year):

> $$\forall t, y, i_1, \cdots,p_1, i_2, \cdots, p_2$$

> $$(\text{Movies}(t, y, i_1, \dots, p_1) \land \text{Movies}(t, y, i_2, \dots, p_2) \rightarrow (i_1 = i_2 \land \dots \land p_1 = p_2))$$

2. Actors (Key: actor_name)

$$\cdots$$

#### C2: Foreign Key Constraints
Ensuring referential integrity between bridge tables and entities.

1.  **Movie_Actors References:**
    >$$ \pi_{\text{title}, \text{year}}(\text{Movie\_Actors}) \subseteq \pi_{\text{title}, \text{year}}(\text{Movies}) $$
    >$$ \pi_{\text{actor\_name}}(\text{Movie\_Actors}) \subseteq \pi_{\text{actor\_name}}(\text{Actors}) $$

2.  **Movie_Genres References:**
    >$$ \pi_{\text{title}, \text{year}}(\text{Movie\_Genres}) \subseteq \pi_{\text{title}, \text{year}}(\text{Movies}) $$


#### C2.2 **Foreign Keys Formal Logic Representation (Inclusion Dependency)**
While referential integrity is expressed via inclusion dependencies in the relational model, it can be formally defined in FOL. For instance,:

1.  **Movie_Actors References:**
> $$\forall t, y, n (\text{Movie\_Actors}(t, y, n) \rightarrow \exists i, v_c, v_a, b, r_w, r_d, p (\text{Movies}(t, y, i, v_c, v_a, b, r_w, r_d, p)))$$
$$\cdots$$

### II. Semantic Constraints (FOL)

#### C3: Domain Rules

1.  **Profit Consistency:**
    Profit should equal worldwide revenue minus budget (where values exist).

    > $$\forall\, t, y, i, v_c, v_a, b, r_w, r_d, p \;(\text{Movies}(t, y, i, v_c, v_a, b, r_w, r_d, p) \rightarrow (p = r_w - b))$$


2.  **Valid Ratings:**

    > $$\forall\, t, y, i, v_c, v_a, b, r, r_d, p \;(\text{Movies}(t, y, i, v_c, v_a, b, r, r_d, p) \rightarrow (0 \leq v_a \leq 10))$$

3.  **Temporal Consistency:**
    Death year must be greater than birth year.

    > $$ \forall n, b, d (\text{Actors}(n, b, d) \rightarrow (d \gt b)) $$

## **Global Schema Queries**

The tasks are ordered by increasing complexity to demonstrate the expressive power and inherent limitations of First-Order Logic.

---

### Task 1: Simple Selection, Projection, and Join

> **Task Goal:** Find the titles of all movies released after the year 2000 that star an actor who was born before 1950.

> $$\{ t \mid \exists y, n, b \, (\text{Movies}(t, y, \dots) \land y > 2000 \land \text{Movie\_Actors}(t, y, n) \land \text{Actors}(n, b, \dots) \land b < 1950) \}$$

---

### Task 2: Complex Query Using Universal Quantification (Minimization)

> **Task Goal:** Find the name and birth year of the actor with the **earliest birth year** (the "Oldest Actor") who has appeared in any "Science Fiction" movie.

> $$\{ n, b \mid \Phi(n, b) \land \forall n', b' \, (\Phi(n', b') \rightarrow b \le b') \}$$

**Where $\Phi(n, b)$ is the candidate formula:**

> $$\Phi(n, b) \equiv \exists t, y \, (\text{Actors}(n, b, \dots) \land \text{Movie\_Actors}(t, y, n) \land \text{Movie\_Genres}(t, y, \text{'Science Fiction'}))$$


---

### Task 3: Complex Query Using Negation (Set Difference)

> **Task Goal:** Find all genres that contain **no movies** with a budget greater than 50 million dollars.

Note: This query is expressed by asserting two conditions: 
   1. *the genre exists in the database*, **AND**
   2. *there does NOT exist a movie in that genre with a budget over $50M.*

> $$\{ g \mid \exists t, y \, (\text{Movie\_Genres}(t, y, g)) \land \neg \exists t', y', b \, (\text{Movie\_Genres}(t', y', g) \land \text{Movies}(t', y', \dots, b, \dots) \land b > 50000000) \}$$

---

### Task 4: Query Demonstrating FOL Limitation (Aggregation)

> **Task Goal:** For every genre, calculate the **average** budget of all movies belonging to that genre.

#### First-Order Logic (FOL)

*Note: Standard First-Order Logic **cannot express aggregation (e.g., AVG, SUM, COUNT)** because it cannot manipulate sets of values or assign a single computed value to a group. The task requires calculating a new value ($\text{average\_budget}$) based on a set of tuples (all movies in a genre), which is outside the scope of FOL's predicate calculus.*

> $$\mathbf{Q_4}(g, \text{avg\_b}) = \text{Impossible to express in FOL using built-in predicates.}$$

#### Relational Algebra (RA)

1. Join the $\text{Movie\_Genres}$ table with the $\text{Movies}$ table to link genres with movie budgets.
   
   > $$\mathbf{R_1} = \text{Movie\_Genres} \bowtie \pi_{\text{title}, \text{year}, \text{budget}}(\text{Movies})$$

2. Group the result by $\text{genre\_name}$ (denoted as $g$) and calculate the average budget for each group.
   
   > $$\mathbf{Q_4} = \gamma_{g, \text{AVG}(\text{budget}) \rightarrow \text{average\_budget}}(\mathbf{R_1})$$

Note: $\large \gamma$ is the RA ***grouping operator***.

## Datasets Overview

In [1]:
import pandas as pd
import numpy as np
import json

In [8]:
df = pd.read_csv('/home/alessio/Documents/ELT_GAV/movie_analytics/data/raw/TMDB_movie_dataset_1M.csv')

In [12]:
df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords'],
      dtype='str')

In [19]:
df[df['title'] == 'Metropolis'][['title', 'release_date', 'vote_count', 'revenue', 'original_title']]

,title,release_date,vote_count,revenue,original_title
1821,Metropolis,1927-02-06,2430,650422,Metropolis
7336,Metropolis,2001-05-26,410,4035192,メトロポリス
495492,Metropolis,NaN,0,0,Metropolis
518400,Metropolis,NaN,0,0,Metropolis
784081,Metropolis,NaN,0,0,Metropolis
1002430,Metropolis,2004-01-01,0,0,Metropolis
1135192,Metropolis,2015-07-13,0,0,Metropolis
1279716,Metropolis,2005-01-01,0,0,Metropolis
1373514,Metropolis,2009-08-07,0,0,Metropolis


### Dataset 1: Box Office

In [5]:
df_box = pd.read_csv('./data/raw/moviesboxoffice.csv')

print(df_box.shape)
print(df_box.columns)
print(df_box.isna().sum())
df_box.head(3)

(5000, 13)
Index(['Rank', 'Release Group', '$Worldwide', '$Domestic', 'Domestic %',
       '$Foreign', 'Foreign %', 'Year', 'Genres', 'Rating', 'Vote_Count',
       'Original_Language', 'Production_Countries'],
      dtype='str')
Rank                      0
Release Group             0
$Worldwide                0
$Domestic                 0
Domestic %                0
$Foreign                  0
Foreign %                 0
Year                      0
Genres                  178
Rating                  170
Vote_Count              170
Original_Language       170
Production_Countries    200
dtype: int64


,Rank,Release Group,$Worldwide,$Domestic,Domestic %,$Foreign,Foreign %,Year,Genres,Rating,Vote_Count,Original_Language,Production_Countries
0,1,Mission: Impossible II,546388108.0,215409889.0,39.4,330978219.0,60.6,2000,"Adventure, Action, Thriller",6.126/10,6741.0,en,United States of America
1,2,Gladiator,460583960.0,187705427.0,40.8,272878533.0,59.2,2000,"Action, Drama, Adventure",8.217/10,19032.0,en,"United Kingdom, United States of America"
2,3,Cast Away,429632142.0,233632142.0,54.4,196000000.0,45.6,2000,"Adventure, Drama",7.663/10,11403.0,en,United States of America


### Dataset 2: TMDB_Movies

In [2]:
df_tmdb = pd.read_csv('/home/alessio/Documents/ELT_GAV/movie_analytics/data/raw/tmdb_5000_movies.csv')

# df_tmdb = df_tmdb.replace(0, np.nan).dropna()

print(df_tmdb.shape)
print(df_tmdb.columns)
print(df_tmdb.replace(0, np.nan).isna().sum())


print(json.loads(df_tmdb['genres'].iloc[0]))

df_tmdb.head()

(4803, 20)
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='str')
budget                  1037
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 1
production_companies       0
production_countries       0
release_date               1
revenue                 1427
runtime                   37
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average              63
vote_count                62
dtype: int64
[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


### Dataset 3: TMDB Cast

In [16]:
import json

df_tmdb_crew = pd.read_csv('./data/raw/tmdb_5000_credits.csv', nrows=10000)

# print(df_tmdb_crew.shape)
# print(df_tmdb_crew.isna().sum())
print(df_tmdb_crew.columns)
# df_tmdb_crew[['cast']].iloc[0]
# json.loads(df_tmdb_crew[['cast']].iloc[0].values[0])
# json.loads(df_tmdb_crew[['crew']].iloc[0].values[0])
# print(df_tmdb_crew[df_tmdb_crew['movie_id'] == 285]['title'])
json.loads(df_tmdb_crew[df_tmdb_crew['movie_id'] == 285][['crew']].iloc[0].values[0])

Index(['movie_id', 'title', 'cast', 'crew'], dtype='str')


[{'credit_id': '52fe4232c3a36847f800b579',
  'department': 'Camera',
  'gender': 2,
  'id': 120,
  'job': 'Director of Photography',
  'name': 'Dariusz Wolski'},
 {'credit_id': '52fe4232c3a36847f800b4fd',
  'department': 'Directing',
  'gender': 2,
  'id': 1704,
  'job': 'Director',
  'name': 'Gore Verbinski'},
 {'credit_id': '52fe4232c3a36847f800b54f',
  'department': 'Production',
  'gender': 2,
  'id': 770,
  'job': 'Producer',
  'name': 'Jerry Bruckheimer'},
 {'credit_id': '52fe4232c3a36847f800b503',
  'department': 'Writing',
  'gender': 2,
  'id': 1705,
  'job': 'Screenplay',
  'name': 'Ted Elliott'},
 {'credit_id': '52fe4232c3a36847f800b509',
  'department': 'Writing',
  'gender': 2,
  'id': 1706,
  'job': 'Screenplay',
  'name': 'Terry Rossio'},
 {'credit_id': '52fe4232c3a36847f800b57f',
  'department': 'Editing',
  'gender': 0,
  'id': 1721,
  'job': 'Editor',
  'name': 'Stephen E. Rivkin'},
 {'credit_id': '52fe4232c3a36847f800b585',
  'department': 'Editing',
  'gender': 2,
 

In [17]:
import json
import pandas as pd

df_tmdb_crew = pd.read_csv('./data/raw/tmdb_5000_credits.csv', nrows=10000)

# Target 'crew' (or 'cast' if custom renamed)
col_name = 'crew' if 'crew' in df_tmdb_crew.columns else 'cast'

departments = set()
jobs = set()

for row in df_tmdb_crew[col_name].dropna():
    crew_list = json.loads(row)
    for item in crew_list:
        if 'department' in item:
            departments.add(item['department'])
        if 'job' in item:
            jobs.add(item['job'])

print(f"Unique Departments ({len(departments)}):", sorted(departments))
print(f"Unique Jobs ({len(jobs)}):", sorted(jobs))

Unique Departments (12): ['Actors', 'Art', 'Camera', 'Costume & Make-Up', 'Crew', 'Directing', 'Editing', 'Lighting', 'Production', 'Sound', 'Visual Effects', 'Writing']
Unique Jobs (418): ['24 Frame Playback', '2D Artist', '2D Supervisor', '3D Animator', '3D Artist', '3D Coordinator', '3D Modeller', '3D Supervisor', 'ADR & Dubbing', 'ADR Editor', 'ADR Supervisor', 'ADR Voice Casting', "Actor's Assistant", 'Adaptation', 'Additional Camera', 'Additional Dialogue', 'Additional Editing', 'Additional Editorial Assistant', 'Additional Music', 'Additional Photography', 'Additional Sound Re-Recording Mixer', 'Additional Soundtrack', 'Additional Still Photographer', 'Additional Writing', 'Administration', 'Aerial Camera', 'Aerial Camera (suggest in addition to Helicopter Camera)', 'Aerial Camera Technician', 'Aerial Coordinator', 'Aerial Director of Photography', 'Ager/Dyer', 'Animal Coordinator', 'Animal Wrangler', 'Animation', 'Animation Department Coordinator', 'Animation Director', 'Animat

In [8]:
json.loads(df_tmdb_crew[['cast']].iloc[0].values[0])


[{'cast_id': 242,
  'character': 'Jake Sully',
  'credit_id': '5602a8a7c3a3685532001c9a',
  'gender': 2,
  'id': 65731,
  'name': 'Sam Worthington',
  'order': 0},
 {'cast_id': 3,
  'character': 'Neytiri',
  'credit_id': '52fe48009251416c750ac9cb',
  'gender': 1,
  'id': 8691,
  'name': 'Zoe Saldana',
  'order': 1},
 {'cast_id': 25,
  'character': 'Dr. Grace Augustine',
  'credit_id': '52fe48009251416c750aca39',
  'gender': 1,
  'id': 10205,
  'name': 'Sigourney Weaver',
  'order': 2},
 {'cast_id': 4,
  'character': 'Col. Quaritch',
  'credit_id': '52fe48009251416c750ac9cf',
  'gender': 2,
  'id': 32747,
  'name': 'Stephen Lang',
  'order': 3},
 {'cast_id': 5,
  'character': 'Trudy Chacon',
  'credit_id': '52fe48009251416c750ac9d3',
  'gender': 1,
  'id': 17647,
  'name': 'Michelle Rodriguez',
  'order': 4},
 {'cast_id': 8,
  'character': 'Selfridge',
  'credit_id': '52fe48009251416c750ac9e1',
  'gender': 2,
  'id': 1771,
  'name': 'Giovanni Ribisi',
  'order': 5},
 {'cast_id': 7,
  'c

In [4]:
df_tmdb_1M = pd.read_csv('./data/raw/TMDB_movie_dataset_1M.csv', nrows = 3)

# print(df_tmdb_1M.shape)
# print(df_tmdb_1M.isna().sum())
# df_tmdb_1M[['title', 'imdb_id']].head(3)
print(df_tmdb_1M.columns)
df_tmdb_1M.head()

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords'],
      dtype='str')


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."


### Dataset 4: IMDB Cast

In [ ]:
# Massive dataset of milions of rows, we take just first 5
imdb_df = pd.read_csv('./data/raw/name.basics.tsv', sep='\t', nrows=5)

imdb_df = imdb_df.repla"Domestic %", "Foreign %", "Rank", "Genres", "Rating"ce('\\N', np.nan)
# print(imdb_df.shape)
# print(imdb_df.isna().sum())
print(imdb_df.columns)
imdb_df.head(3)

Index(['nconst', 'primaryName', 'birthYear', 'deathYear', 'primaryProfession',
       'knownForTitles'],
      dtype='str')


,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0117057"
2,nm0000003,Brigitte Bardot,1934,2025,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"


Only Directors

In [34]:
# Massive dataset of milions of rows, we take just first 5
imdb_df = pd.read_csv('./data/raw/name.basics.tsv', sep='\t', nrows=100000)

imdb_df = imdb_df.replace('\\N', np.nan)
# print(imdb_df.shape)
# print(imdb_df.isna().sum())
imdb_df[imdb_df['primaryProfession'].str.split(',').str[0] == 'director']
# imdb_df.head(3)

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
32,nm0000033,Alfred Hitchcock,1899,1980,"director,producer,writer","tt0054215,tt0052357,tt0053125,tt0034248"
87,nm0000088,Aleksey Korenev,1927,1995,"director,writer,assistant_director","tt0100069,tt0191104,tt0068304,tt0076551"
90,nm0000091,Gérard Pirès,1942,NaN,"director,writer,actor","tt0152930,tt0064297,tt0282552,tt0421974"
179,nm0000180,David Lean,1908,1991,"director,writer,editor","tt0056172,tt0059113,tt0050212,tt0038574"
230,nm0000231,Oliver Stone,1946,NaN,"director,producer,writer","tt0091763,tt0096969,tt0102138,tt0110632"
...,...,...,...,...,...,...
99821,nm0104978,Nicholas Brandt,1966,NaN,"director,special_effects",NaN
99837,nm0104994,Robert Brandt,1925,2009,"director,writer,producer","tt0050200,tt0047093,tt0059009,tt0064524"
99868,nm0105027,David Brandvik,NaN,NaN,"director,producer,writer","tt3190238,tt0194321,tt1019442,tt0228414"
99938,nm0105099,Myriam Braniff,1960,1997,"director,writer,producer","tt0106294,tt0390316,tt0230282,tt0110324"


## Pre-processing

In [ ]:
import pandas as pd
import json
import os
import numpy as np

# Define the base directory for the input files
BASE_DIR = "/home/alessio/Documents/ELT_GAV/data/raw"
OUTPUT_DIR = "/home/alessio/Documents/ELT_GAV/data/preprocessed"

# Define input file paths
BOX_OFFICE_PATH = os.path.join(BASE_DIR, "Movies Box Office/movies_box_office.csv")
TMDB_MOVIES_PATH = os.path.join(BASE_DIR, "TMDB Movie Dataset/tmdb_5000_movies.csv")
TMDB_CAST_PATH = os.path.join(BASE_DIR, "TMDB Movie Dataset/tmdb_5000_credits.csv")
IMDB_CAST_PATH = os.path.join(BASE_DIR, "IMDB_Dataset/name.basics.tsv")

# Helper function to ensure directory exists
def ensure_dir(file_path):
    directory = os.path.dirname(file_path)
    if not os.path.exists(directory):
        os.makedirs(directory)

def normalize_box_office(df):
    """
    Normalizes the Box_Office dataset.
    Schema: Box_Office(title, year, revenue_worldwide, revenue_domestic, revenue_foreign)
    PK: (title, year)
    """
    print("-> Normalizing Box Office...")

    # Rename columns to match the Source Schema (S)
    df = df.rename(columns={
        'Release Group': 'title',
        'Year': 'year',
        '$Worldwide': 'revenue_worldwide',
        '$Domestic': 'revenue_domestic',
        '$Foreign': 'revenue_foreign'
    })

    # Clean revenue columns (remove '$' and ',')
    for col in ['revenue_worldwide', 'revenue_domestic', 'revenue_foreign']:
        df[col] = df[col].astype(str).str.replace(r'[$,]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # UPDATE: Convert to nullable integer type Int64
        df[col] = df[col].astype("Int64")

    # Convert year to integer
    df['year'] = pd.to_numeric(df['year'], errors='coerce', downcast='integer').astype("Int64")

    # Select required attributes
    final_cols = ['title', 'year', 'revenue_worldwide', 'revenue_domestic', 'revenue_foreign']
    df_normalized = df[final_cols].copy()

    # Constraint Check: PK (title, year) must be unique and not null
    df_normalized.dropna(subset=['title', 'year'], inplace=True)
    df_normalized.drop_duplicates(subset=['title', 'year'], keep='first', inplace=True)

    print(f"   -> Box Office Normalized Rows: {len(df_normalized)}")
    return df_normalized

def normalize_tmdb_movies(df):
    """
    Normalizes the TMDB_Movies dataset.
    Schema: TMDB_Movies(movie_id, title, year, budget, revenue, vote_count, vote_average)
    PK: movie_id
    """
    print("-> Normalizing TMDB Movies (Core Attributes)...")

    # Rename 'id' to 'movie_id'
    df = df.rename(columns={'id': 'movie_id'})

    # Extract year from release_date
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    df['year'] = df['release_date'].dt.year
    # UPDATE: Convert year to nullable integer type Int64
    df['year'] = df['year'].astype("Int64")

    # Select required attributes
    final_cols = ['movie_id', 'title', 'year', 'budget', 'revenue', 'vote_count', 'vote_average']
    df_normalized = df[final_cols].copy()

    # Constraint Check: PK (movie_id) must be unique and not null
    df_normalized.dropna(subset=['movie_id'], inplace=True)
    df_normalized['movie_id'] = pd.to_numeric(df_normalized['movie_id'], errors='coerce', downcast='integer').astype("Int64")
    df_normalized['year'] = pd.to_numeric(df_normalized['year'], errors='coerce', downcast='integer').astype("Int64")

    df_normalized.drop_duplicates(subset=['movie_id'], keep='first', inplace=True)

    print(f"   -> TMDB Movies Normalized Rows: {len(df_normalized)}")
    return df_normalized

def normalize_tmdb_genres(df):
    """
    Normalizes TMDB data to create the TMDB_Genres bridge table.
    Schema: TMDB_Genres(movie_id, genre_name)
    PK: (movie_id, genre_name)
    """
    print("-> Flattening TMDB Genres (creating TMDB_Genres)...")
    
    # Prepare base dataframe
    df_genres = df.rename(columns={'id': 'movie_id'})[['movie_id', 'genres']].copy()

    def extract_genres(genres_json):
        if pd.isna(genres_json) or genres_json == '[]':
            return []
        try:
            genres_list = json.loads(genres_json)
            return [{'genre_name': genre['name']} for genre in genres_list]
        except (json.JSONDecodeError, TypeError):
            return []

    df_genres['genres_list'] = df_genres['genres'].apply(extract_genres)
    
    # Explode to create one row per genre
    df_exploded = df_genres.explode('genres_list').dropna(subset=['genres_list']).reset_index(drop=True)

    if not df_exploded.empty:
        df_exploded['genre_name'] = df_exploded['genres_list'].apply(lambda x: x.get('genre_name') if isinstance(x, dict) else None)
        df_normalized = df_exploded[['movie_id', 'genre_name']].copy()
    else:
        df_normalized = pd.DataFrame(columns=['movie_id', 'genre_name'])
        
    # Constraint Check: PK (movie_id, genre_name)
    # Ensure movie_id is nullable integer
    df_normalized['movie_id'] = pd.to_numeric(df_normalized['movie_id'], errors='coerce', downcast='integer').astype("Int64")
    df_normalized.dropna(subset=['movie_id', 'genre_name'], inplace=True)
    df_normalized.drop_duplicates(subset=['movie_id', 'genre_name'], keep='first', inplace=True)

    print(f"   -> TMDB Genres Flattened Rows: {len(df_normalized)}")
    return df_normalized

def normalize_tmdb_cast(df):
    """
    Normalizes TMDB data to create the TMDB_Cast bridge table.
    Schema: TMDB_Cast(movie_id, actor_name)
    PK: (movie_id, actor_name)
    """
    print("-> Flattening TMDB Cast (creating TMDB_Cast)...")

    def extract_actors(cast_json):
        if pd.isna(cast_json):
            return []
        try:
            cast_list = json.loads(cast_json)
            # Extract actor name
            return [{'actor_name': actor['name']} for actor in cast_list]
        except (json.JSONDecodeError, TypeError):
            return []

    df['actors'] = df['cast'].apply(extract_actors)

    # Explode to create one row per actor
    df_exploded = df.explode('actors').dropna(subset=['actors']).reset_index(drop=True)

    if not df_exploded.empty:
        df_exploded['actor_name'] = df_exploded['actors'].apply(lambda x: x.get('actor_name') if isinstance(x, dict) else None)
        df_normalized = df_exploded[['movie_id', 'actor_name']].copy()
        # UPDATE: Convert movie_id to nullable integer Int64
        df_normalized['movie_id'] = pd.to_numeric(df_normalized['movie_id'], errors='coerce').astype('Int64')
    else:
        df_normalized = pd.DataFrame(columns=['movie_id', 'actor_name'])
        
    # Constraint Check: PK (movie_id, actor_name)
    df_normalized.dropna(subset=['movie_id', 'actor_name'], inplace=True)
    df_normalized.drop_duplicates(subset=['movie_id', 'actor_name'], keep='first', inplace=True)

    print(f"   -> TMDB Cast Flattened Rows: {len(df_normalized)}")
    return df_normalized

def normalize_imdb_cast():
    """
    Normalizes the IMDB_Cast dataset.
    Schema: IMDB_Cast(actor_name, birth_year, death_year)
    PK: actor_name
    """
    print("-> Normalizing IMDB Cast...")

    dtype_map = {
        'nconst': str,
        'primaryName': str,
        'birthYear': str,
        'deathYear': str
    }
    
    df = pd.read_csv(IMDB_CAST_PATH, sep='\t', na_values='\\N', dtype=dtype_map)

    # Clean years first
    df['birthYear'] = pd.to_numeric(df['birthYear'], errors='coerce')
    df['deathYear'] = pd.to_numeric(df['deathYear'], errors='coerce')

    # Drop rows where biographical info is completely missing (optional, but good for quality)
    df.dropna(subset=['birthYear', 'deathYear'], how='all', inplace=True)

    # Convert to standard Int64 (nullable int)
    df['birthYear'] = df['birthYear'].astype("Int64")
    df['deathYear'] = df['deathYear'].astype("Int64")

    # Rename columns to match Source Schema
    df = df.rename(columns={
        'primaryName': 'actor_name', 
        'birthYear': 'birth_year', 
        'deathYear': 'death_year'
    })

    # Select final columns
    final_cols = ['actor_name', 'birth_year', 'death_year']
    df_normalized = df[final_cols].copy()

    # Constraint Check: PK (actor_name) must be unique
    # Since IMDB has multiple people with same name, we drop duplicates to adhere 
    # to our "Unique Name Assumption" defined in the project theory.
    df_normalized.dropna(subset=['actor_name'], inplace=True)
    df_normalized.drop_duplicates(subset=['actor_name'], keep='first', inplace=True)
    
    print(f"   -> IMDB Cast Normalized Rows: {len(df_normalized)}")
    return df_normalized


# --- Main Execution ---

try:
    # 1. Box Office
    output_path_box = os.path.join(OUTPUT_DIR, "BoxOffice_Movies/movies_box_office.csv")
    ensure_dir(output_path_box)
    
    box_office_df = pd.read_csv(BOX_OFFICE_PATH)
    box_office_normalized = normalize_box_office(box_office_df)
    box_office_normalized.to_csv(output_path_box, index=False)
    print(f"   -> Saved to {output_path_box}")

    # 2. TMDB Movies (Core Attributes)
    output_path_tmdb = os.path.join(OUTPUT_DIR, "TMDB_Movies/tmdb_5000_movies.csv")
    ensure_dir(output_path_tmdb)

    tmdb_movies_df = pd.read_csv(TMDB_MOVIES_PATH)
    tmdb_movies_normalized = normalize_tmdb_movies(tmdb_movies_df.copy()) 
    tmdb_movies_normalized.to_csv(output_path_tmdb, index=False)
    print(f"   -> Saved to {output_path_tmdb}")

    # 3. TMDB Genres (Flattened)
    output_path_genres = os.path.join(OUTPUT_DIR, "TMDB_Genres/tmdb_genres.csv")
    ensure_dir(output_path_genres)

    tmdb_genres_normalized = normalize_tmdb_genres(tmdb_movies_df.copy())
    tmdb_genres_normalized.to_csv(output_path_genres, index=False)
    print(f"   -> Saved to {output_path_genres}")

    # 4. TMDB Cast (Flattened)
    output_path_cast = os.path.join(OUTPUT_DIR, "TMDB_Cast/tmdb_5000_credits.csv")
    ensure_dir(output_path_cast)

    tmdb_cast_df = pd.read_csv(TMDB_CAST_PATH)
    tmdb_cast_normalized = normalize_tmdb_cast(tmdb_cast_df)
    tmdb_cast_normalized.to_csv(output_path_cast, index=False)
    print(f"   -> Saved to {output_path_cast}")

    # 5. IMDB Cast
    output_path_imdb = os.path.join(OUTPUT_DIR, "IMDB_Cast/name.basics.tsv")
    ensure_dir(output_path_imdb)

    imdb_cast_normalized = normalize_imdb_cast()
    # Note: Saving as CSV with tabs for consistency with original format, or just CSV
    imdb_cast_normalized.to_csv(output_path_imdb, sep='\t', index=False)
    print(f"   -> Saved to {output_path_imdb}")
    
    print("\n✅ All datasets have been normalized and saved to the './Global/' directory.")

except FileNotFoundError as e:
    print(f"\n❌ Error: One or more input files were not found.")
    print(f"   Please ensure the following path is correct: {e.filename}")
    print(f"   The script expects the files in the directory structure: {BASE_DIR}")
except Exception as e:
    print(f"\n❌ An unexpected error occurred during processing: {e}")

-> Normalizing Box Office...
   -> Box Office Normalized Rows: 4999
   -> Saved to ./Global/BoxOffice_Movies/movies_box_office.csv
-> Normalizing TMDB Movies (Core Attributes)...
   -> TMDB Movies Normalized Rows: 4803
   -> Saved to ./Global/TMDB_Movies/tmdb_5000_movies.csv
-> Flattening TMDB Genres (creating TMDB_Genres)...
   -> TMDB Genres Flattened Rows: 12160
   -> Saved to ./Global/TMDB_Genres/tmdb_genres.csv
-> Flattening TMDB Cast (creating TMDB_Cast)...
   -> TMDB Cast Flattened Rows: 106076
   -> Saved to ./Global/TMDB_Cast/tmdb_5000_credits.csv
-> Normalizing IMDB Cast...
   -> IMDB Cast Normalized Rows: 650213
   -> Saved to ./Global/IMDB_Cast/name.basics.tsv

✅ All datasets have been normalized and saved to the './Global/' directory.


## Results Check

### Box Office Preprocessing

Preprocessing removed unused columns and renamed revenue's columns:
- `Release Group` to `title`
- `Year` to `year`
- `$Worldwide, $Domestic, $Foreign` to `revenue_worldwide, revenue_domestic, revenue_foreign`

In [ ]:
box_office = pd.read_csv('./Global/BoxOffice_Movies/movies_box_office.csv')

print(box_office.columns, box_office.shape)
print(box_office.isna().sum())

box_office['year'].head(3)

Index(['title', 'year', 'revenue_worldwide', 'revenue_domestic',
       'revenue_foreign'],
      dtype='object') (4999, 5)
title                0
year                 0
revenue_worldwide    0
revenue_domestic     0
revenue_foreign      0
dtype: int64


0    2000
1    2000
2    2000
Name: year, dtype: int64

### Movies TMDB Preprocessing

Preprocessing dropped unused columns and converted the "release date" column to "year", which is then used as part of the PK of Movies table rows.

In [3]:
movies_tmdb = pd.read_csv('./Global/TMDB_Movies/tmdb_5000_movies.csv')

print(movies_tmdb.columns, movies_tmdb.shape)
print(movies_tmdb.isna().sum())

movies_tmdb.head(3)

Index(['movie_id', 'title', 'year', 'budget', 'revenue', 'vote_count',
       'vote_average'],
      dtype='object') (4803, 7)
movie_id        0
title           0
year            1
budget          0
revenue         0
vote_count      0
vote_average    0
dtype: int64


,movie_id,title,year,budget,revenue,vote_count,vote_average
0,19995,Avatar,2009.0,237000000,2787965087,11800,7.2
1,285,Pirates of the Caribbean: At World's End,2007.0,300000000,961000000,4500,6.9
2,206647,Spectre,2015.0,245000000,880674609,4466,6.3


### TMDB Cast Preprocessing

Preprocessing flattened the json `cast` column from original dataset and created single tuples `movie_id, actor_name`.

It also removed unused columns

In [ ]:
actors = pd.read_csv('./Global/TMDB_Cast/tmdb_5000_credits.csv')

print(actors.columns, actors.shape)
print(actors.isna().sum())

actors.head(3)

Index(['movie_id', 'actor_name'], dtype='object') (106076, 2)
movie_id      0
actor_name    0
dtype: int64


,movie_id,actor_name
0,19995,Sam Worthington
1,19995,Zoe Saldana
2,19995,Sigourney Weaver


### IMDB Cast Preprocessing

In [ ]:
imdb_actors = pd.read_csv('./Global/IMDB_Cast/name.basics.tsv', sep='\t')

print(imdb_actors.columns, imdb_actors.shape)
print(imdb_actors.isna().sum())

imdb_actors.head(3)

Index(['actor_name', 'birth_year', 'death_year'], dtype='object') (650213, 3)
actor_name         0
birth_year     14921
death_year    407635
dtype: int64


,actor_name,birth_year,death_year
0,Fred Astaire,1899.0,1987.0
1,Lauren Bacall,1924.0,2014.0
2,Brigitte Bardot,1934.0,NaN


### TMDB Genres Preprocessing

Preprocessing extracts from 'TMDB_Movies' dataset, for each film, tuples `movie_id, genre_name`

In [ ]:
genres = pd.read_csv('./Global/TMDB_Genres/tmdb_genres.csv')

print(genres.columns, genres.shape)
print(genres.isna().sum())

genres.drop_duplicates(subset=['genre_name'])

Index(['movie_id', 'genre_name'], dtype='object') (12160, 2)
movie_id      0
genre_name    0
dtype: int64


,movie_id,genre_name
0,19995,Action
1,19995,Adventure
2,19995,Fantasy
3,19995,Science Fiction
9,206647,Crime
12,49026,Drama
13,49026,Thriller
20,38757,Animation
21,38757,Family
44,57201,Western
